(tutorial_compressor_models_label)=

# Choose a compressor model

The useful level of detail depends on the question and on the available machine
data. A fixed efficiency can be enough for a design-point energy balance.
Offdesign work needs information on how the machine changes away from that
point. A characteristic line, a turbocompressor map and a displacement-machine
polynomial describe different assumptions; they are not an accuracy ranking.

## Select the representation

| Representation | Information supplied | What it determines |
|---|---|---|
| {py:class}`~tespy.components.turbomachinery.compressor.Compressor` with fixed `eta_s` | Isentropic efficiency and a pressure or power specification | The outlet state and power from a single efficiency; efficiency does not change with load. |
| `Compressor` with `eta_s_char` | Design efficiency and a one-dimensional relative efficiency characteristic, normally against relative mass flow | Efficiency follows the characteristic in offdesign; pressure ratio still needs another specification. |
| {py:class}`~tespy.components.turbomachinery.turbocompressor.TurboCompressor` | Design state and pressure-ratio/efficiency maps over corrected flow and speed lines | Pressure ratio and efficiency are coupled through the maps; guide-vane angle can be a control variable. |
| {py:class}`~tespy.components.displacementmachinery.polynomial_compressor.PolynomialCompressor` | Refrigerant performance data or EN 12900 polynomials, reference superheat/subcooling, frequency and displacement | Isentropic and volumetric efficiencies determine the outlet state and throughput of a displacement machine. |
| {py:class}`~tespy.components.displacementmachinery.polynomial_compressor_with_cooling.PolynomialCompressorWithCooling` | The polynomial model plus a cooling circuit and heat-recovery efficiency | The cooling-fluid heat gain is included in the network as well as working-fluid compression. |

Use measured or manufacturer data for the intended machine and operating range.
Adding a generic characteristic to a model does not validate it for a particular
compressor.

## Compare fixed efficiency, a part-load characteristic and a map

We compress air from 1 bar and 20 °C to 5 bar. The design inlet volume flow is
50 l/s and the design isentropic efficiency is 0.8 for all three representations.
Only the inlet flow changes below. The built-in characteristic and maps are
generic TESPy data used to demonstrate the workflow.

For the map case, the inlet temperature and shaft-speed assumption are unchanged,
and the guide-vane angle is solved to maintain the specified pressure ratio.
Its curve therefore includes that control action; it is not an uncontrolled
fixed-guide-vane operating line.


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from tespy.components import Compressor, Sink, Source, TurboCompressor
from tespy.connections import Connection
from tespy.networks import Network


def make_air_network(representation):
    network = Network(iterinfo=False)
    network.units.set_defaults(
        pressure="bar", pressure_difference="bar", temperature="degC",
        volumetric_flow="l/s", power="kW",
    )
    compressor_class = TurboCompressor if representation == "map" else Compressor
    compressor = compressor_class("air compressor", char_warnings=False)
    inlet = Connection(Source("air inlet"), "out1", compressor, "in1")
    outlet = Connection(compressor, "out1", Sink("air outlet"), "in1")
    network.add_conns(inlet, outlet)
    inlet.set_attr(fluid={"air": 1}, p=1, T=20, v=50)
    compressor.set_attr(pr=5, eta_s=0.8)
    if representation == "characteristic":
        compressor.set_attr(design=["eta_s"], offdesign=["eta_s_char"])
    elif representation == "map":
        compressor.set_attr(
            design=["eta_s"], offdesign=["char_map_pr", "char_map_eta_s"],
        )
    network.solve("design")
    network.assert_convergence()
    design_state = network.save(as_dict=True)
    if representation == "map":
        compressor.set_attr(igva="var")
    return network, compressor, inlet, outlet, design_state


def check_air_solution(network, compressor, inlet, outlet):
    network.assert_convergence()
    assert np.isfinite([compressor.P.val, compressor.eta_s.val]).all()
    assert 0 < compressor.eta_s.val <= 1
    assert np.isclose(inlet.m.val_SI, outlet.m.val_SI)
    assert np.isclose(outlet.p.val_SI / inlet.p.val_SI, 5)
    assert np.isclose(
        compressor.P.val_SI,
        inlet.m.val_SI * (outlet.h.val_SI - inlet.h.val_SI),
    )


A saved design state supplies the reference efficiency, flow, pressure and
temperature needed to normalize the offdesign relations. Keep `eta_s` fixed for
the constant-efficiency case. For the other two cases, `design=["eta_s"]` and
the corresponding `offdesign` settings switch from the design efficiency to the
characteristic equations.


In [ ]:
representations = {
    "fixed": "Fixed efficiency",
    "characteristic": "Part-load characteristic",
    "map": "Turbocompressor map",
}
air_rows = []
for representation, label in representations.items():
    network, compressor, inlet, outlet, design_state = make_air_network(
        representation,
    )
    for relative_flow in np.arange(80, 106, 5) / 100:
        inlet.set_attr(v=50 * relative_flow)
        if representation == "fixed":
            network.solve("design")
        else:
            network.solve("offdesign", design_path=design_state)
        check_air_solution(network, compressor, inlet, outlet)
        air_rows.append({
            "Model": label,
            "Inlet flow / design": relative_flow,
            "Isentropic efficiency": compressor.eta_s.val,
            "Power (kW)": compressor.P.val,
            "Guide-vane angle (deg)": compressor.igva.val,
        })

air_results = pd.DataFrame(air_rows)
design_points = air_results[np.isclose(air_results["Inlet flow / design"], 1)]
assert np.allclose(design_points["Isentropic efficiency"], 0.8)
assert np.allclose(design_points["Power (kW)"], design_points["Power (kW)"].iloc[0])
fixed_results = air_results[air_results["Model"] == "Fixed efficiency"]
assert np.allclose(fixed_results["Isentropic efficiency"], 0.8)
air_results.pivot(
    index="Inlet flow / design", columns="Model", values="Isentropic efficiency",
).round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for label, results in air_results.groupby("Model", sort=False):
    flow_percent = 100 * results["Inlet flow / design"]
    axes[0].plot(flow_percent, results["Isentropic efficiency"], "o-", label=label)
    axes[1].plot(flow_percent, results["Power (kW)"], "o-", label=label)
for axis in axes:
    axis.set_xlabel("Inlet volume flow / design (%)")
    axis.grid(alpha=0.3)
axes[0].set_ylabel("Isentropic efficiency")
axes[1].set_ylabel("Compressor power (kW)")
axes[0].legend()
plt.show()


The three models agree at the chosen design point because they share the same
efficiency there. Away from it, constant `eta_s` remains 0.8, the one-dimensional
characteristic changes efficiency with load, and the map couples efficiency
and pressure ratio with the guide-vane setting. The map is not necessarily
closer to a real machine unless its data and control assumptions match it.

For example, fixing the guide-vane angle instead would require releasing
another specification, such as the inlet flow, so the map can determine the
operating point. See the `TurboCompressor` class example for that alternative.

## Why polynomial compressors are a different model

A turbocompressor transfers energy through rotating blades. Its maps describe
pressure rise and efficiency using non-dimensional inlet flow and corrected
speed lines. A positive-displacement compressor transports a volume per
revolution. The polynomial classes therefore also need a **volumetric
efficiency** and a displacement/frequency reference to determine mass flow.
They use efficiencies fitted against evaporation and condensation temperatures,
rather than the turbocompressor's corrected-flow/speed map.

The next example uses the R134a manufacturer dataset already provided in
TESPy's `PolynomialCompressor` class example, sourced there from
{cite:p}`bitzer2025_HSK`. Its reference state is part of the dataset, not an
optional unit conversion. The data below use watts for power and cooling
capacity. Frequency scaling follows the model's displacement relation; it does
not add a measured speed-dependent efficiency map.

This is a separate refrigerant displacement-machine example. Its absolute
power or efficiency must not be compared with the air-machine curves above as
if both represented the same compressor.


In [ ]:
from CoolProp.CoolProp import PropsSI

from tespy.components import PolynomialCompressor, PolynomialCompressorWithCooling
from tespy.components.displacementmachinery.polynomial_compressor import (
    generate_eta_polys_from_data,
)

reference_state = {
    "T_sh": 20, "T_sc": 0, "frequency_poly": 50.0,
    "displacement": 214, "frequency_displacement": 20.0,
}
evaporation_temperatures = [10, 7.5, 5, 0, -5, -10]
cooling_data = pd.DataFrame(
    [
        [465600, 424100, 385500, 316700, 257900, 208000],
        [418900, 380400, 344800, 281400, 227400, 181600],
        [365900, 331300, 299200, 242100, 193700, 152900],
    ],
    index=[30, 40, 50], columns=evaporation_temperatures,
)
power_data = pd.DataFrame(
    [
        [62.0, 61.8, 61.8, 61.8, 61.7, 61.3],
        [78.0, 78.0, 78.0, 78.0, 77.7, 76.8],
        [99.2, 99.2, 99.2, 98.9, 98.1, 96.5],
    ],
    index=[30, 40, 50], columns=evaporation_temperatures,
) * 1000
eta_s_poly, eta_vol_poly = generate_eta_polys_from_data(
    power_data, cooling_data, "R134a", reference_state,
)


In [ ]:
def make_polynomial_network(with_cooling=False):
    network = Network(iterinfo=False)
    network.units.set_defaults(
        pressure="bar", pressure_difference="bar", temperature="degC", power="kW",
    )
    compressor_class = (
        PolynomialCompressorWithCooling if with_cooling else PolynomialCompressor
    )
    compressor = compressor_class("refrigerant compressor")
    inlet = Connection(Source("from evaporator"), "out1", compressor, "in1")
    outlet = Connection(compressor, "out1", Sink("to condenser"), "in1")
    network.add_conns(inlet, outlet)
    inlet.set_attr(fluid={"R134a": 1}, T=0, td_dew=10)
    compressor.set_attr(
        eta_s_poly=eta_s_poly, eta_vol_poly=eta_vol_poly,
        dissipation_ratio=0.05, reference_state=reference_state.copy(), frequency=20,
    )
    cooling_inlet = cooling_outlet = None
    if with_cooling:
        cooling_inlet = Connection(
            Source("cooling water inlet"), "out1", compressor, "in2",
        )
        cooling_outlet = Connection(
            compressor, "out2", Sink("cooling water outlet"), "in1",
        )
        network.add_conns(cooling_inlet, cooling_outlet)
        cooling_inlet.set_attr(fluid={"water": 1}, T=20, p=1)
        cooling_outlet.set_attr(T=40)
        compressor.set_attr(eta_recovery=0.9, dp_cooling=0)
    return network, compressor, inlet, outlet, cooling_inlet, cooling_outlet


polynomial_rows = []
for with_cooling in [False, True]:
    network, compressor, inlet, outlet, cooling_inlet, cooling_outlet = (
        make_polynomial_network(with_cooling)
    )
    for condensing_temperature in [30, 40, 50]:
        outlet.set_attr(
            p=PropsSI("P", "Q", 0, "T", condensing_temperature + 273.15, "R134a")
            / 1e5,
        )
        network.solve("design")
        network.assert_convergence()
        assert 0 < compressor.eta_s.val <= 1
        assert 0 < compressor.eta_vol.val <= 1
        assert np.isclose(inlet.m.val_SI, outlet.m.val_SI)
        assert np.isclose(
            compressor.P.val_SI + compressor.Q_diss.val_SI,
            inlet.m.val_SI * (outlet.h.val_SI - inlet.h.val_SI),
        )
        recovered_heat = 0.0
        if with_cooling:
            recovered_heat = cooling_inlet.m.val_SI * (
                cooling_outlet.h.val_SI - cooling_inlet.h.val_SI
            )
            assert np.isclose(
                recovered_heat,
                -compressor.Q_diss.val_SI * compressor.eta_recovery.val_SI,
            )
        polynomial_rows.append({
            "Model": "With cooling" if with_cooling else "Without cooling",
            "Condensing temperature (degC)": condensing_temperature,
            "R134a flow (kg/s)": inlet.m.val_SI,
            "Isentropic efficiency": compressor.eta_s.val,
            "Volumetric efficiency": compressor.eta_vol.val,
            "Power (kW)": compressor.P.val,
            "Recovered heat (kW)": recovered_heat / 1000,
        })

polynomial_results = pd.DataFrame(polynomial_rows)
polynomial_results.round(3)


At fixed frequency, changing the pressure lift changes both fitted
efficiencies. The volumetric efficiency changes the refrigerant throughput,
which would have to be specified or otherwise constrained in a basic
`Compressor` model. This is why changing class is more than replacing one
efficiency curve.

The cooling variant keeps working-fluid `in1`/`out1` and adds cooling-fluid
`in2`/`out2` connections. Here 90% of the assumed dissipated heat is recovered
by water warming from 20 to 40 °C; its required flow is a solver result.
The ordinary polynomial compressor also accounts for the specified
dissipation, but does not transfer recovered heat into a second network stream.

## Before choosing a model

- Match the machine type and available data first. A displacement polynomial
  cannot stand in for a turbocompressor's surge/choke map.
- Fit and use performance data inside its supported temperature, flow and
  speed range. Do not assume a generic characteristic or polynomial remains
  valid under extrapolation.
- Choose pressure, flow, speed and guide-vane specifications consistently:
  the network must leave the map or displacement relation something to solve.
- Include the cooling variant when the cooling circuit or recovered heat is
  part of the system question. Use the class examples linked in the overview
  for additional parameter and reference-state details.
